# Password Strenght Classification

<img src='https://imgs.xkcd.com/comics/password_strength_2x.png'>

🇹🇷 Türkçe Bu proje, kullanıcıların şifrelerini güvenlik seviyelerine göre sınıflandırmayı amaçlamaktadır. Veri setindeki Strength etiketi, 0 (zayıf), 1 (orta), 2 (güçlü) ve 3 (çok güçlü) olmak üzere 4 sınıf içerir ve şifrelerin içerdiği rakamlar, özel karakterler ve uzunluk gibi özelliklere dayalı olarak belirlenir. Çalışmada, makine öğrenmesi ve/veya derin öğrenme algoritmaları kullanılarak şifrelerin güvenlik seviyelerini tahmin eden bir sınıflandırma modeli geliştirilecektir.

🇬🇧 English This project aims to classify users' passwords according to their security level. The Strength label in the dataset includes three classes: 0 (weak), 1 (medium), 2 (strong) and 3 (very strong) and is determined based on features such as the numbers, special characters, and length of the passwords. In this study, a classification model will be developed that predicts the security levels of passwords using machine learning and/or deep learning algorithms.

In [1]:
import pandas as pd
pd.set_option('display.max_columns',100)

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import math
from collections import Counter
import math
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder


from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
df_list = [
    pd.read_csv('pwlds_weak.csv'),
    pd.read_csv('pwlds_average.csv'),
    pd.read_csv('pwlds_strong.csv'),
    pd.read_csv('pwlds_very_strong.csv')
]

df = pd.concat(df_list, ignore_index=True)

In [3]:
df.head()

,Password,Strength_Level
0,Activate9999,1
1,bbbbadjurer4,1
2,asdfadinole5,1
3,acetoinAsdf,1
4,qweabsurdly3,1


In [4]:
df.tail()

,Password,Strength_Level
12001044,dR'.StP9VYudR4uE,4
12001045,(*d^yLm}]/f5P6EE)'</WCDlbYu,4
12001046,z7%G]UHWWjC0}@Q?vGfD<KtSTRBov9wr,4
12001047,G.Q%h[l1kWh2(A/dz'#@`XX,4
12001048,>4qB^r4~Z/MBJcF>EIt1x,4


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12001049 entries, 0 to 12001048
Data columns (total 2 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   Password        object
 1   Strength_Level  int64 
dtypes: int64(1), object(1)
memory usage: 183.1+ MB


In [6]:
df.isnull().sum()

Password          0
Strength_Level    0
dtype: int64

In [7]:
from collections import Counter
import math

def extract_features(Password):
    """Her şifre için 8 temel sayısal özellik ekleyelim"""
    length = len(Password)
    if length == 0:
        return {k: 0 for k in ['length', 'unique_ratio', 'digits', 'upper', 'lower', 'special', 'entropy', 'char_types']}
    
    digits = sum(c.isdigit() for c in Password)
    upper = sum(c.isupper() for c in Password)
    lower = sum(c.islower() for c in Password)
    special = sum(not c.isalnum() for c in Password)
    
    counts = Counter(Password)  
    entropy = -sum((count/length) * math.log2(count/length) for count in counts.values())
    
    return {
        'length': length,
        'unique_ratio': len(set(Password)) / length,
        'digits': digits,
        'upper': upper,
        'lower': lower,
        'special': special,
        'entropy': entropy,
        'char_types': sum([digits>0, upper>0, lower>0, special>0])
    }

In [8]:
features_df = pd.DataFrame(df['Password'].apply(extract_features).tolist())
df = pd.concat([df, features_df], axis=1)

In [9]:
df.head()

,Password,Strength_Level,length,unique_ratio,digits,upper,lower,special,entropy,char_types
0,Activate9999,1,12,0.666667,4,1,7,0,2.751629,3
1,bbbbadjurer4,1,12,0.666667,1,0,11,0,2.751629,2
2,asdfadinole5,1,12,0.833333,1,0,11,0,3.251629,2
3,acetoinAsdf,1,11,1.000000,0,1,10,0,3.459432,2
4,qweabsurdly3,1,12,1.000000,1,0,11,0,3.584963,2


In [30]:
x = df.drop(["Password", "Strength_Level"], axis=1, errors='ignore')
y = df["Strength_Level"]

In [11]:
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12001049 entries, 0 to 12001048
Data columns (total 8 columns):
 #   Column        Dtype  
---  ------        -----  
 0   length        int64  
 1   unique_ratio  float64
 2   digits        int64  
 3   upper         int64  
 4   lower         int64  
 5   special       int64  
 6   entropy       float64
 7   char_types    int64  
dtypes: float64(2), int64(6)
memory usage: 732.5 MB


In [12]:
x_train,x_test, y_train, y_test=train_test_split(x,y, random_state=42, test_size=0.20)

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

b = BernoulliNB()
l = LogisticRegression()
d = DecisionTreeClassifier()
r = RandomForestClassifier()
gb= GradientBoostingClassifier()
kn= KNeighborsClassifier()
ab= AdaBoostClassifier()
mn= MultinomialNB()

def algo_test(x, y):
    modeller=[ b, l, d, r, gb, kn, ab, mn]
    isimler=["BernoulliNB", "LogisticRegression", "DecisionTreeClassifier", 
             "RandomForestClassifier", "GradientBoostingClassifier", "KNeighborsClassifier",
             "AdaBoostClassifier", "MultinomialNB"]

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=.20, random_state = 42)
    
    accuracy = []
    precision = []
    recall = []
    f1 = []
    mdl=[]

    print("Veriler hazır modeller deneniyor")
    for model in modeller:
        print(model, " modeli eğitiliyor!..")
        model=model.fit(x_train,y_train)
        tahmin=model.predict(x_test)
        mdl.append(model)
        accuracy.append(accuracy_score(y_test, tahmin))
        precision.append(precision_score(y_test, tahmin, average="micro"))
        recall.append(recall_score(y_test, tahmin, average="micro"))
        f1.append(f1_score(y_test, tahmin, average="micro"))
        print(confusion_matrix(y_test, tahmin))

    print("Eğitim tamamlandı.")
    
    metrics=pd.DataFrame(columns=["Accuracy", "Precision", "Recall", "F1", "Model"], index=isimler)
    metrics["Accuracy"] = accuracy
    metrics["Precision"] = precision  
    metrics["Recall"] = recall
    metrics["F1"] = f1
    metrics["Model"]=mdl

    metrics.sort_values("F1", ascending=False, inplace=True)

    print("En başarılı model: ", metrics.iloc[0].name)
    model=metrics.iloc[0,-1]
    tahmin=model.predict(np.array(x_test) if model==kn else x_test)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, tahmin))
    print("classification Report:")
    print(classification_report(y_test, tahmin))
    print("Diğer Modeller:")
    
    return metrics.drop("Model", axis=1)

In [14]:
algo_test(x,y)

Veriler hazır modeller deneniyor
BernoulliNB()  modeli eğitiliyor!..
[[600229      0      0      0]
 [ 58913 540902      0      0]
 [152318 252227  39508 155939]
 [   152  46827   1394 551801]]
LogisticRegression()  modeli eğitiliyor!..
[[552831  28005  18682    711]
 [ 34825 529026  35962      2]
 [ 79697 106436 409420   4439]
 [     1      0   4733 595440]]
DecisionTreeClassifier()  modeli eğitiliyor!..
[[566170  12156  21903      0]
 [ 16881 555936  26998      0]
 [ 22028  84079 493885      0]
 [     0      0      0 600174]]
RandomForestClassifier()  modeli eğitiliyor!..
[[566277  12043  21909      0]
 [ 17027 555790  26998      0]
 [ 22016  84078 493898      0]
 [     0      0      0 600174]]
GradientBoostingClassifier()  modeli eğitiliyor!..
[[567025   9754  23450      0]
 [ 20346 552750  26719      0]
 [ 23430  84773 491789      0]
 [     0      0      0 600174]]
KNeighborsClassifier()  modeli eğitiliyor!..
[[563035  18390  18804      0]
 [ 20622 519122  60071      0]
 [ 31146  8

,Accuracy,Precision,Recall,F1
DecisionTreeClassifier,0.923321,0.923321,0.923321,0.923321
RandomForestClassifier,0.923310,0.923310,0.923310,0.923310
GradientBoostingClassifier,0.921477,0.921477,0.921477,0.921477
KNeighborsClassifier,0.904416,0.904416,0.904416,0.904416
AdaBoostClassifier,0.881531,0.881531,0.881531,0.881531
LogisticRegression,0.869389,0.869389,0.869389,0.869389
MultinomialNB,0.737807,0.737807,0.737807,0.737807
BernoulliNB,0.721787,0.721787,0.721787,0.721787


In [31]:
## Classification with Deep Learning

In [32]:
x.head()

,length,unique_ratio,digits,upper,lower,special,entropy,char_types
0,12,0.666667,4,1,7,0,2.751629,3
1,12,0.666667,1,0,11,0,2.751629,2
2,12,0.833333,1,0,11,0,3.251629,2
3,11,1.000000,0,1,10,0,3.459432,2
4,12,1.000000,1,0,11,0,3.584963,2


In [33]:
y.value_counts()

Strength_Level
3    3000975
1    3000048
2    3000026
4    3000000
Name: count, dtype: int64

In [34]:
# Label encoding
le = LabelEncoder()
y= le.fit_transform(y)
num_classes = len(le.classes_)

In [35]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, random_state=42, stratify=y)

In [36]:
from sklearn.preprocessing import StandardScaler

# Scale et (fit sadece train'e, transform ikisine)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test  = scaler.transform(x_test)  

In [37]:
# Model
model = Sequential()
model.add(Dense(128, activation='relu'))
model.add(Dense(64, activation='relu'))
model.add(Dense(30, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [38]:
early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

In [39]:
history=model.fit(x,y,batch_size=32, validation_split=0.20,verbose=1, epochs=5,callbacks=[early_stop])

Epoch 1/5
300027/300027 ━━━━━━━━━━━━━━━━━━━━ 1169s 4ms/step - accuracy: 0.8982 - loss: 0.2406 - val_accuracy: 0.9999 - val_loss: 3.0535e-04
Epoch 2/5
300027/300027 ━━━━━━━━━━━━━━━━━━━━ 1180s 4ms/step - accuracy: 0.9018 - loss: 0.2304 - val_accuracy: 0.9999 - val_loss: 0.0014
Epoch 3/5
300027/300027 ━━━━━━━━━━━━━━━━━━━━ 1171s 4ms/step - accuracy: 0.9018 - loss: 0.2306 - val_accuracy: 0.9999 - val_loss: 2.6416e-04
Epoch 4/5
300027/300027 ━━━━━━━━━━━━━━━━━━━━ 1651s 6ms/step - accuracy: 0.9015 - loss: 0.2321 - val_accuracy: 0.9997 - val_loss: 0.0014
Epoch 5/5
300027/300027 ━━━━━━━━━━━━━━━━━━━━ 1864s 6ms/step - accuracy: 0.9009 - loss: 0.2361 - val_accuracy: 0.9999 - val_loss: 0.0135


In [40]:
loss, accuracy=model.evaluate(x,y)

375033/375033 ━━━━━━━━━━━━━━━━━━━━ 1019s 3ms/step - accuracy: 0.9204 - loss: 0.1840


In [41]:
# Tahmin 
tahmin = model.predict(x_test)
tahmin_classes = tahmin.argmax(axis=1)

75007/75007 ━━━━━━━━━━━━━━━━━━━━ 144s 2ms/step


In [43]:
model.save('password_model.h5')

Parola güçlülüğü sınıflandırma probleminde farklı makine öğrenmesi algoritmaları karşılaştırılmış ve en yüksek performansın DecisionTreeClassifier ve RandomForestClassifier ile yaklaşık %92.33 doğruluk oranı ile elde edildiği görülmüştür. GradientBoostingClassifier ve KNeighborsClassifier modelleri de benzer şekilde yüksek performans göstermiştir. Derin öğrenme modeli ise %92.31 doğruluk ve 0.18 loss değeri ile klasik makine öğrenmesi modellerine oldukça yakın bir sonuç üretmiştir. Genel olarak, bu problemde geleneksel ağaç tabanlı yöntemlerin en başarılı yaklaşım olduğu gözlemlenmiştir.

In the password strength classification problem, different machine learning algorithms were compared, and it was observed that the highest performance was obtained with DecisionTreeClassifier and RandomForestClassifier, with an accuracy rate of approximately 92.33%. GradientBoostingClassifier and KNeighborsClassifier models also showed similarly high performance. The deep learning model, on the other hand, produced a result quite close to classical machine learning models with an accuracy of 92.31% and a loss value of 0.18. Overall, it was observed that traditional tree-based methods were the most successful approach in this problem.
